In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CUDA unavailable')

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, time, zipfile
archive = Path('/content/xray_colab_t4.zip')
expected_sha = 'fb5c1c1f69fd24e9d1c00cd4d5407261ceb838ae2cd83a30deaeac0412401b9b'
saved_archive = Path('/content/drive/MyDrive/xray_overnight_20261003/xray_colab_t4.zip')
if not archive.is_file() and saved_archive.is_file():
    shutil.copyfile(saved_archive, archive)
assert archive.is_file(), 'Upload xray_colab_t4.zip in the Files panel first'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected_sha, 'Upload incomplete or file mismatch'
project = Path('/content/xray_project')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    assert all((project / n).resolve().is_relative_to(project) for n in z.namelist())
    z.extractall(project)
checks = json.loads((project / 'bundle_checksums.json').read_text())
for name, expected in checks.items():
    assert hashlib.sha256((project / name).read_bytes()).hexdigest() == expected, name
storage = Path('/content/drive/MyDrive/xray_overnight_20261003')
assert storage.parent.is_dir(), 'Mount Drive first'
storage.mkdir(exist_ok=True)
backup = storage / archive.name
if not backup.exists() or hashlib.sha256(backup.read_bytes()).hexdigest() != expected_sha:
    shutil.copyfile(archive, backup)
print('Verified', len(checks), 'files. Package backed up to Drive.', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
env_dir = Path('/content/xray_env312')
python = str(env_dir / 'bin/python')
if not Path(python).exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12.14', str(env_dir)], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', python, 'torch==2.8.0', 'torchvision==0.23.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', python, 'numpy==1.26.4', 'pandas==2.3.3', 'Pillow==11.3.0', 'opencv-python-headless==4.11.0.86', 'PyYAML==6.0.2'], check=True)
subprocess.run([python, '-c', "import sys,torch,torchvision; print('READY',sys.version,torch.__version__,torchvision.__version__,torch.cuda.get_device_name(0)); assert torch.cuda.is_available()"], check=True)
print('Setup complete. Run the next cell to train and evaluate.', flush=True)

In [ ]:

import subprocess
subprocess.run([python, '-u', 'scripts/xray_colab_run.py', '--storage', str(storage)], cwd=project, check=True)